# Time Series Comparison

This notebook pulls in data from the NOAA CO-OPS NWLON stations and pulls in CORA data at those locations from the NOAA Open Data Dissemination (NODD) and plots water level time series for comparison.

## Configuration

Update these values as necessary

`station_id` is a list of NOAA ID's. One source of information is https://tidesandcurrents.noaa.gov/map/index.html?region=Texas

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  CONFIGURATION - MODIFY THESE VALUES AS NEEDED
# ═══════════════════════════════════════════════════════════════

# NOAA ID's of stations to query
station_id = [
              '8665530', # Charleston, SC
              '8774770', # Port Aransas, TX
              '8771450', # Galveston Pier 21, TX
              '8775237', # Aransas, Aransas Pass, TX
              '8775296', # Enbridge, Ingleside, TX
              '8773037', # Seadrift, TX
              '8776604', # Baffin Bay, TX
              '8775792', # Packery Channel, TX
              '8773146', # Matagorda City, TX
              '8770613', # Morgans Point, TX
              '8770777', # Manchester, TX
              '8770822', # Texas Point, TX
              '8770475', # Port Arthur, TX
              '8779770', # Port Isabel, TX
              '8736897', # Coast Guard Sector Mobile, AL
            ]

#, '8775870']


# add US coast guard, baffin bay, packery channel, madagorda city, morgans point, manchester, texas point, port arthur,

# Time period for data extraction
start_year = '2018'
start_month = '09'
start_day = '01'

end_year = '2018'
end_month = '11'
end_day = '01'

## Import python libraries.

In [ ]:
import requests
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import dask
import intake
import math
# import xarray as xr
import scipy.spatial as sp
# import s3fs
# import geopy.distance
from scipy.spatial import KDTree
# import folium
# from folium import Marker
# from folium.plugins import HeatMap, MarkerCluster
# from branca.colormap import linear, LinearColormap
from tqdm import tqdm
from dask.diagnostics import ProgressBar
import os

### Define Utility functions
DO NOT MODIFY

In [ ]:
def area(x1, y1, x2, y2, x3, y3):
    return (abs((x1 * (y2 - y3) + x2 * (y3 - y1)
                + x3 * (y1 - y2)) / 2.0))

def define_kd_tree(ds):
    e = ds.element.values.astype(int)
    emin1 = e-1
    num_elems = len(e)
    x_vals = ds.x.values
    y_vals = ds.y.values

    xe=np.mean(x_vals[emin1],axis=1)
    ye=np.mean(y_vals[emin1],axis=1)
    tree = sp.KDTree(np.c_[xe,ye])
    areas = [area(x_vals[emin1[k][0]],y_vals[emin1[k][0]],\
                  x_vals[emin1[k][1]],y_vals[emin1[k][1]],\
                  x_vals[emin1[k][2]],y_vals[emin1[k][2]])for k in range(0, num_elems)]
    return tree, areas, e, x_vals, y_vals


def find_triangle(x_vals, y_vals, e,lat,lon):
    e = ds.element.values.astype(int)-1

    k = 10
    dist, ii = tree.query([lon,lat],k=k)
    ii = ii
    triangle_i = -1

    for i in range(0,k):

      a1 = area(lon,lat,\
                x_vals[e[ii[i]][0]],y_vals[e[ii[i]][0]],\
                x_vals[e[ii[i]][1]],y_vals[e[ii[i]][1]])

      a2 = area(lon,lat,\
                x_vals[e[ii[i]][1]],y_vals[e[ii[i]][1]],\
                x_vals[e[ii[i]][2]],y_vals[e[ii[i]][2]])

      a3 = area(lon,lat,\
                x_vals[e[ii[i]][0]],y_vals[e[ii[i]][0]],\
                x_vals[e[ii[i]][2]],y_vals[e[ii[i]][2]])

      t_area = a1 + a2 + a3
      if abs(t_area - areas[ii[i]]) < 0.00000001:
        triangle_i = ii[i]+1
        break
    if(triangle_i == -1):
        print("ERROR for " ,lat,lon)
    return triangle_i


def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the great circle distance between two points on Earth
    using the Haversine formula. Returns distance in meters.
    """
    # Convert latitude and longitude from degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])

    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))

    # Radius of Earth in meters
    r = 6371000

    return c * r

def check_nodes_have_data(node_indices, ds, sample_size=10):
    """
    Check if nodes have valid (non-NaN) water level data by sampling a few time points
    """
    try:
        # Sample a few time points to check for data availability
        sample_data = ds["zeta"].isel(time=slice(0, sample_size), node=node_indices).compute()
        # Check if all data is NaN
        all_nan = np.isnan(sample_data.values).all()
        return not all_nan
    except:
        return False

def find_best_nodes_with_fallback(station_lat, station_lng, ds, tree, areas, e, x_vals, y_vals, k=3):
    """
    Enhanced node finding with fallback strategy:
    1. Try to find triangle containing the station
    2. If no triangle, try nearest nodes and check if they have valid data
    3. If nearest nodes don't have data, fall back to nearest water nodes

    Returns: nodes, distances_m, method_used, fallback_info
    """
    all_depths = ds.depth.values

    # Method 1: Try to find containing triangle
    triangle_result = find_triangle(x_vals, y_vals, e, station_lat, station_lng)

    if triangle_result != -1:
        # Found a triangle - use its nodes
        triangle_nodes = e[triangle_result-1]

        # Check if triangle nodes have data
        if check_nodes_have_data(triangle_nodes, ds):
            # Calculate distances
            distances_m = []
            for node_idx in triangle_nodes:
                node_lat = y_vals[node_idx]
                node_lng = x_vals[node_idx]
                dist = haversine_distance(station_lat, station_lng, node_lat, node_lng)
                distances_m.append(dist)

            return triangle_nodes + 1, distances_m, "triangle", None

    # Method 2: Try nearest nodes
    node_coords = np.c_[x_vals, y_vals]
    kdtree = KDTree(node_coords)
    distances_deg, nearest_indices = kdtree.query([station_lng, station_lat], k=k)
    nearest_nodes = nearest_indices + 1

    # Convert degree distances to meters
    distances_m = []
    for i, node_idx in enumerate(nearest_indices):
        node_lat = y_vals[node_idx]
        node_lng = x_vals[node_idx]
        dist = haversine_distance(station_lat, station_lng, node_lat, node_lng)
        distances_m.append(dist)

    # Check if nearest nodes have valid data
    if check_nodes_have_data(nearest_indices, ds):
        return nearest_nodes, distances_m, "nearest", None

    # Method 3: Fallback to nearest water nodes
    print(f"    Nearest nodes don't have valid data")
    nearest_depths = all_depths[nearest_indices]
    print(f"    Nearest node depths: {nearest_depths}")
    print(f"    Falling back to nearest water nodes...")

    # Filter to only water nodes (positive depth)
    water_mask = all_depths > 0
    water_coords = node_coords[water_mask]
    water_indices = np.where(water_mask)[0]

    # Build KDTree for water nodes only
    water_kdtree = KDTree(water_coords)
    water_distances_deg, water_nearest_indices = water_kdtree.query([station_lng, station_lat], k=k)

    # Convert back to original node indices
    water_nodes = water_indices[water_nearest_indices] + 1

    # Calculate actual distances in meters
    water_distances_m = []
    for i, water_idx in enumerate(water_indices[water_nearest_indices]):
        node_lat = y_vals[water_idx]
        node_lng = x_vals[water_idx]
        dist = haversine_distance(station_lat, station_lng, node_lat, node_lng)
        water_distances_m.append(dist)

    # Prepare fallback info with all land node distances
    fallback_info = {
        'land_distances_m': distances_m,  # All land node distances
        'nearest_water_distance_m': water_distances_m[0],
        'extra_distance_m': water_distances_m[0] - distances_m[0],
        'land_depths': nearest_depths
    }

    return water_nodes, water_distances_m, "water_fallback", fallback_info


## Get CORA dataset files
**Access the data on the NODD and initialize the available CORA datasets.** 

*This accesses a .yml file located on the NODD that shows which CORA output files are available to import.*

In [ ]:
# @title This accesses a .yml file located on the NODD that shows which CORA output files are available to import.
catalog = intake.open_catalog("s3://noaa-nos-cora-pds/CORA_V1.1_intake.yml",storage_options={'anon':True})
list(catalog)

***CORA-V1.1-fort.63:*** Hourly water levels: ***'zeta'*** is the water elevation variable referenced to mean sea level<br>
***CORA-V1.1-swan_DIR.63:*** Hourly mean wave direction: ***'swan_DIR'*** is the mean wave direction variable<br>
***CORA-V1.1-swan_TPS.63:*** Hourly peak wave periods: ***'swan_TPS'*** is the peak wave period variable<br>
***CORA-V1.1-swan_HS.63:*** Hourly significant wave heights: ***'swan_HS'*** is the significant wave height variable<br>
***CORA-V1.1-Grid:*** Hourly water levels interpolated from model nodes to 500-meter resolution coastal grid: ***'zeta'*** is the water elevation variable <br>

> All datasets denoted as **'-timeseries'** are optimized for pulling long time series (greater than a few days)
> For up to a few days of data, use the regular dataset (not labeled **'-timeseries'** in the catalog description)


*Now, create an xarray dataset for the CORA data that you would like to use.*<br>
<br>
Using the `.to_dask()` command with the water level dataset located in the catalog will create an xarray dataset.
There is data at 1,813,443 model nodes spanning 385,704 hours (44 years, 1979-2022).
The 'zeta' hourly water level variable is given in dimensions of time and node.


In [ ]:
ds = catalog["CORA-V1.1-fort.63"].to_dask()

In [ ]:
# stationpointsfile = 'C:\\Users\\John.Ratcliff\\CORA\\HSOFS_GEC_Stations.csv'
# cora_stations = pd.read_csv(stationpointsfile)
# cora_stations['id'] = cora_stations['id'].astype(str)
# cora_stations.iloc[62]

## Get NWLON station data

**Create a dataframe of NWLON station ids and coordinates from the CO-OPS API where you want to do a comparison.**

In [ ]:
base_url = 'https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations/.json'
params = {
    'type': 'waterlevels',
    'units': 'metric'
}
print(f'base_url: {base_url}, parameters: {params}')
response = requests.get(base_url, params=params)
content = response.json()

stations = content['stations']
stations_df = pd.DataFrame(stations)

# Include station name along with id, lat, lng
stations_df = stations_df[['id','name','lat','lng','state']]

# limit to the list of stations specified in the configuration section
stations_df = stations_df[stations_df['id'].isin(station_id)]

stations_df

**Loop through the station list to grab water level hourly heights for each station. Create a pandas dataframe of the time series data.**

In [ ]:
base_url = 'https://api.tidesandcurrents.noaa.gov/api/prod/datagetter'

# Initialize an empty dataframe to merge all station data
df = None

for i in range(len(stations_df)):
    station_id = stations_df.id.iloc[i]
    print(f"Requesting data for station {station_id}")

    params = {
        'begin_date': f'{start_year}{start_month}{start_day}',
        'end_date': f'{end_year}{end_month}{end_day}',
        'station': station_id,
        'product': 'hourly_height',
        'datum': 'MSL',
        'time_zone': 'gmt',
        'units': 'metric',
        'format': 'json'
    }

    print(f'base_url: {base_url}, parameters: {params}')

    response = requests.get(base_url, params=params)
    content = response.json()

    # Check if we have data in the response
    if 'data' not in content or not content['data']:
        print(f"No data available for station {station_id}")
        continue

    # Extracting data for 'time' and 'height'
    time_data = [d['t'] for d in content['data']]
    tnc_data = [d['v'] for d in content['data']]

    # Convert tnc_data water level to numeric
    tnc_data = [float(x) if x and str(x).strip() != '' else np.nan for x in tnc_data]

    # Create a temporary dataframe for this station
    temp_df = pd.DataFrame({
        'time': time_data,
        f'tnc_{station_id}': tnc_data
    })

    # Convert time to datetime and set as index
    temp_df['time'] = pd.to_datetime(temp_df['time'])
    temp_df.set_index('time', inplace=True)

    # Merge with the main dataframe
    if df is None:
        df = temp_df
    else:
        df = df.merge(temp_df, left_index=True, right_index=True, how='outer')

# Display info about the final dataframe
print(f"Final dataframe shape: {df.shape}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
df

## Calculate CORA water levels

### Get CORA node Coordinates

In [ ]:
tree, areas, e, x_vals, y_vals = define_kd_tree(ds)

In [ ]:

# Extract node coordinates from the mesh
node_coords = np.c_[ds.x.values, ds.y.values]
kdtree = KDTree(node_coords)

# Initialize storage
elem = np.zeros((len(stations_df), 1), dtype=int)
elem_nodes = np.zeros((len(stations_df), 3), dtype=int)
distances_meters = np.zeros((len(stations_df), 3))
weights = np.zeros((len(stations_df), 3), dtype=float)
station_methods = []  # Track which method was used for each station
station_fallback_info = []  # Track fallback information

print("Processing stations...")

for i in tqdm(range(len(stations_df)), desc="Processing stations"):
    station_lat = stations_df.lat.iloc[i]
    station_lng = stations_df.lng.iloc[i]
    station_id = stations_df.id.iloc[i]

    print(f"\n--- Processing Station {station_id} ---")
    print(f"Station coordinates: Lat={station_lat}, Lng={station_lng}")

    # Use enhanced node finding
    nodes, distances_m, method, fallback_info = find_best_nodes_with_fallback(
        station_lat, station_lng, ds, tree, areas, e, x_vals, y_vals
    )

    elem_nodes[i,:] = nodes
    distances_meters[i,:] = distances_m
    station_methods.append(method)
    station_fallback_info.append(fallback_info)

    print(f"Method used: {method}")
    print(f"Selected nodes: {nodes}")
    print(f"Distances: {[f'{d:.0f}m' for d in distances_m]}")

    if fallback_info:
        land_distances = fallback_info.get('land_distances_m', [])
        land_distance_text = ', '.join([f'{d:.0f}m' for d in land_distances])
        print(f"Fallback info: Land nodes at {land_distance_text}, went to water nodes at {fallback_info['nearest_water_distance_m']:.0f}m")
        print(f"Extra distance: {fallback_info['extra_distance_m']:.0f}m")

    # Calculate weights based on distances (inverse distance weighting)
    if np.any(np.array(distances_m) == 0):
        # Exact match at a node
        weights[i,:] = np.where(np.array(distances_m) == 0, 1, 0)
    else:
        # Inverse distance weighting
        inv_distances = 1.0 / np.array(distances_m)
        weights[i,:] = inv_distances / np.sum(inv_distances)

    print(f"Weights: {weights[i,:]}")

unique_nodes = np.unique(elem_nodes)
mapped_triangle = np.searchsorted(unique_nodes, elem_nodes)

print(f"\nSummary:")
print(f"Total unique nodes needed: {len(unique_nodes)}")
print(f"Methods used: {dict(zip(stations_df.id, station_methods))}")

# Store method info for later use in plotting
stations_df['extraction_method'] = station_methods
stations_df['fallback_info'] = station_fallback_info

In [ ]:
unique_nodes[mapped_triangle]

In [ ]:
concat_nodes=np.concatenate(node_coords[elem_nodes-1], axis=0)
lat_nodes=concat_nodes[:,1]
lon_nodes=concat_nodes[:,0]

df_nodes=pd.DataFrame({'Lon': lon_nodes, 'Lat': lat_nodes})
df_nodes

In [ ]:
# disabled for now. It's not particularly useful

# # Create a base map centered on the first coordinate
# map_center = [stations_df['lat'].iloc[0], stations_df['lng'].iloc[0]]
# my_map = folium.Map(location=map_center, zoom_start=8)

# # Add markers for each coordinate
# for index, row in df_nodes.iterrows():
#     folium.CircleMarker(
#         location=[row['Lat'], row['Lon']],
#     ).add_to(my_map)

# for index, row in stations_df.iterrows():
#     folium.CircleMarker(
#         location=[row['lat'], row['lng']],
#         popup=row['id'], color='red'
#     ).add_to(my_map)

# tile = folium.TileLayer(
#         tiles = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
#         attr = 'Esri',
#         name = 'Esri Satellite',
#         overlay = False,
#         control = True
#        ).add_to(my_map)

# # Save the map to an HTML file
# # my_map.save("nodesmap.html")

# my_map

### Calculate CORA water levels

**Average the water levels at the 3 nodes of the element containing the station coordinates for the selected time period. Append these CORA values as new columns to the dataframe.**

In [ ]:
################################################################################
# Original version. Keeping for reference
################################################################################

# %%time

# start_t = f'{start_year}-{start_month}-{start_day} 00:00:00'
# end_t = f'{end_year}-{end_month}-{end_day} 23:00:00'
# dt_range=pd.date_range(start_t, end_t, freq='h',inclusive='both')

# num_ts = len(dt_range) # number of time samples
# t = np.zeros((num_ts, 3), dtype=float) # preallocate with zeros
# zeta_point = np.zeros((num_ts), dtype=float) # preallocate with zeros
# mean_zeta = np.zeros((num_ts,len(stations_df)), dtype=float)

# # for i in range(len(stations_df)):
# for i in tqdm(range(len(stations_df)), desc="Processing stations"):
#     zeta_tslice = ds["zeta"].sel(time=slice(start_t, end_t), node=unique_nodes[mapped_triangle[i]]-1).compute()
#     t = zeta_tslice.values * weights[i]
#     zeta_point = np.sum(t, axis=1) / np.sum(weights[i])
#     mean_zeta_i = np.nanmean(zeta_tslice.values, axis=1)

#     zeta_point[np.isnan(zeta_point)] = mean_zeta_i[np.isnan(zeta_point)]

#     df['cora_'+stations_df.iloc[i,0]] = zeta_point


In [ ]:
%%time

################################################################################
# Optimized version
################################################################################

start_t = f'{start_year}-{start_month}-{start_day} 00:00:00'
end_t = f'{end_year}-{end_month}-{end_day} 23:00:00'

# Get all unique nodes needed across all stations
all_needed_nodes = []
for i in range(len(stations_df)):
    all_needed_nodes.extend(unique_nodes[mapped_triangle[i]]-1)
all_needed_nodes = np.unique(all_needed_nodes)

# Extract data for all nodes at once
print("Extracting data for all stations...")
with ProgressBar():
    zeta_all = ds["zeta"].sel(time=slice(start_t, end_t), node=all_needed_nodes).compute()

# Pre-allocate results array
results = np.zeros((len(zeta_all.time), len(stations_df)))

# Process each station using the pre-loaded data
for i in range(len(stations_df)):
    # Find which indices in zeta_all correspond to this station's nodes
    station_nodes = unique_nodes[mapped_triangle[i]]-1
    node_indices = np.searchsorted(all_needed_nodes, station_nodes)

    # Extract just this station's data
    station_data = zeta_all.values[:, node_indices]

    # Apply weights
    weighted_data = station_data * weights[i]
    zeta_point = np.sum(weighted_data, axis=1) / np.sum(weights[i])

    # Handle NaN values
    mean_zeta_i = np.nanmean(station_data, axis=1)
    zeta_point[np.isnan(zeta_point)] = mean_zeta_i[np.isnan(zeta_point)]

    # Store result
    results[:, i] = zeta_point

# Add all columns to dataframe at once
for i, station_id in enumerate(stations_df.id):
    df[f'cora_{station_id}'] = results[:, i]


## Plot the data

**Plot the time series data for the NWLON observations and CORA for comparison.**

In [ ]:
ylabel = "MSL, m"

# Create subdirectory for plots
plot_dir = "plots"
os.makedirs(plot_dir, exist_ok=True)

for i in range(len(stations_df)):
    # Create title using both station ID and name
    station_id = stations_df.iloc[i, 0]  # id column
    station_name = stations_df.iloc[i, 1]  # name column
    station_state = stations_df.iloc[i, 4]  # state column
    title = f"Hourly Water Levels for {station_name}, {station_state}: {station_id}"

    # Create larger figure
    plt.figure(figsize=(15, 8))
    # print(plt.rcParams.keys())

    df[['tnc_'+stations_df.id.iloc[i],'cora_'+stations_df.iloc[i,0]]].plot(figsize=(15, 8))

    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('Date/Time (GMT)', fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.legend(['NOAA Observed', 'CORA Model'], loc='upper right')

    # Add fallback information annotation from the node extraction results
    if i < len(station_methods):  # Check if we have method info for this station
        method = station_methods[i]
        station_distances = distances_meters[i] if i < len(distances_meters) else []
        station_fallback = station_fallback_info[i] if i < len(station_fallback_info) else {}
        distance_text = ', '.join([f"{d:.0f}m" for d in station_distances]) if len(station_distances) > 0 else 'N/A'

        if method == 'triangle':
            # Station was inside a triangle - ideal case
            annotation_text = "Station location found within mesh triangle"
            annotation_text += f"\nDistance: {distance_text}"
            annotation_color = 'lightgreen'
        elif method == 'nearest':
            # Station near mesh but not in triangle - good case
            annotation_text = f"Using nearest water nodes\nDistance: {distance_text}"
            annotation_color = 'lightblue'
        elif method == 'water_fallback':
            # Station required water fallback - show all land node distances
            land_distances = station_fallback.get('land_distances_m', [])
            land_distance_text = ', '.join([f"{d:.0f}m" for d in land_distances]) if land_distances else 'N/A'
            annotation_text = f"⚠️ Water node fallback required\nLand nodes at: {land_distance_text}\nWater nodes at: {distance_text}"
            annotation_color = 'lightyellow'
        else:
            annotation_text = f"Method: {method}\nDistance: {distance_text}"
            annotation_color = 'lightgray'

        # Add annotation box in upper left corner
        plt.annotate(annotation_text,
                    xy=(0.02, 0.98), xycoords='axes fraction',
                    verticalalignment='top', horizontalalignment='left',
                    bbox=dict(boxstyle='round,pad=0.4', facecolor=annotation_color, alpha=0.8),
                    fontsize=10, fontweight='normal')

    plt.tight_layout()

    # Create filename with time range in YYYY-MM format and save to subdirectory
    time_range = f"{start_year}-{start_month}_to_{end_year}-{end_month}"
    filename = f"{station_id}_{time_range}_water_levels.png"

    # Create plots directory if it doesn't exist
    plot_dir = "plots"
    if not os.path.exists(plot_dir):
        os.makedirs(plot_dir)

    filepath = os.path.join(plot_dir, filename)
    plt.savefig(filepath, dpi=300, bbox_inches='tight')
    print(f"Saved plot: {filepath}")

plt.show()